<a href="https://colab.research.google.com/github/BenayaL/Cloud_Computing_Project/blob/main/Tut3_mqtt_exampleCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **שימו לב - ברגע שאתם מעלים את המחברת לגיטהאב שלכם תמחקו את ה key!**



In [ ]:
pip install paho-mqtt

In [ ]:
import paho.mqtt.client as mqtt
import json

# Callback function for when a message is received
def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode())

        # Extract values with lowercase field names
        temperature = data.get("temperature", "N/A")
        humidity = data.get("humidity", "N/A")
        soil= data.get("soil", "N/A")
        print(f"Temperature: {temperature}°C, Humidity: {humidity}%")

    except json.JSONDecodeError:
        print("Received invalid JSON data")
        # MQTT setup
broker = "io.adafruit.com"
username = "braude6"   #insert Adafruit user name
aio_key = "*******************" #insert key from Adafruit
topic = f"{username}/feeds/json"
client = mqtt.Client()
client.username_pw_set(username,aio_key)
client.on_message = on_message

client.connect(broker, 1883, 60)
client.subscribe(topic)

print(f"Subscribed to MQTT topic: {topic}")
client.loop_forever()



In [ ]:
import requests

USERNAME = "braude6"   #insert Adafruit user name
AIO_KEY = "*******************" #insert key from Adafruit
FEED = "json"

url = f"https://io.adafruit.com/api/v2/{USERNAME}/feeds/{FEED}/data"
headers = {"X-AIO-Key": AIO_KEY}

response = requests.get(url, headers=headers)
data = response.json()

for item in data[:50]:  # חמשת הנתונים האחרונים
    print(f"Value: {item['value']}, Time: {item['created_at']}")


In [ ]:
import requests
import matplotlib.pyplot as plt
from datetime import datetime
import json

USERNAME = "braude6"
AIO_KEY = "*******************"
FEED = "json"

url = f"https://io.adafruit.com/api/v2/{USERNAME}/feeds/{FEED}/data"
headers = {"X-AIO-Key": AIO_KEY}

params = {"limit": 100}

response = requests.get(url, headers=headers, params=params)
data = response.json()

data = data[::-1]

values = []
times = []

for item in data:
    try:
        payload = json.loads(item["value"])

        values.append(payload["temperature"])  # or "humidity"
        times.append(datetime.fromisoformat(item["created_at"].replace("Z", "+00:00")))

    except Exception as e:
        print("Skipping:", item["value"], e)

plt.figure()

plt.plot(times, values, marker="o")

plt.title("Temperature - Last 100 Feed Values")
plt.xlabel("Time")
plt.ylabel("Temperature (°C)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()